# Part 7: LOYO 模型对比

请先运行 `01_dependencies_and_data.ipynb`，然后在 `02_rf`、`03_gbdt`、`04_ann`、`05_Seasonal-LSTM`、`06_lstm` 中运行 LOYO 模块，生成各模型的 LOYO 结果 CSV。

### LOYO 结果对比（Leave-One-Year-Out Cross-Validation）

加载各模型的 LOYO 结果 CSV，汇总对比平均 R²、RMSE、MAE，并绘制逐年 R² 对比图。  
需先运行各模型 notebook 中的 LOYO 模块（`RUN_LOYO=True`）。

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

root_dir    = "C:\\ML4GM"
outputs_dir = os.path.join(root_dir, "models")

# ---- 加载各模型 LOYO CSV ----
loyo_files = {
    "RF":              "rf_loyo_results.csv",
    "LightGBM":        "lgbm_loyo_results.csv",
    "MLP/ANN":         "ann_loyo_results.csv",
    "Seasonal-LSTM":   "lstm_loyo_results.csv",
    "Temporal-LSTM":   "temporal_lstm_loyo_results.csv",
}

loyo_dfs = {}
for model_name, fname in loyo_files.items():
    fpath = os.path.join(outputs_dir, fname)
    if os.path.exists(fpath):
        loyo_dfs[model_name] = pd.read_csv(fpath)
        print(f"[OK] {model_name}: loaded {fpath}")
    else:
        print(f"[SKIP] {model_name}: {fpath} not found — run the model's LOYO section first.")

# ---- 汇总表 ----
summary_rows = []
for model_name, df in loyo_dfs.items():
    summary_rows.append({
        "Model":     model_name,
        "R2_mean":   round(df["R2"].mean(),   4),
        "R2_std":    round(df["R2"].std(),    4),
        "RMSE_mean": round(df["RMSE"].mean(), 4),
        "RMSE_std":  round(df["RMSE"].std(),  4),
        "MAE_mean":  round(df["MAE"].mean(),  4),
        "MAE_std":   round(df["MAE"].std(),   4),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("R2_mean", ascending=False).reset_index(drop=True)
print("\n===== LOYO Summary =====")
print(summary_df.to_string(index=False))
summary_df.to_csv(os.path.join(outputs_dir, "loyo_summary.csv"), index=False)

In [ ]:
# ---- 图1: 平均 R²/RMSE/MAE 柱状图（含误差棒）----
if summary_df is not None and len(summary_df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), dpi=100)
    colors = ["#2196F3", "#FF9800", "#4CAF50", "#9C27B0", "#F44336"]

    # R²
    axes[0].bar(summary_df["Model"], summary_df["R2_mean"],
                yerr=summary_df["R2_std"], capsize=5,
                color=colors[:len(summary_df)], alpha=0.85)
    for i, row in summary_df.iterrows():
        axes[0].text(i, row["R2_mean"] + row["R2_std"] + 0.005,
                     f'{row["R2_mean"]:.4f}', ha="center", va="bottom", fontsize=11)
    axes[0].set_title("LOYO Mean R² (±std)", fontsize=14)
    axes[0].set_ylabel("R²", fontsize=12)
    axes[0].set_ylim(0, 1)
    axes[0].tick_params(axis="x", labelsize=10, rotation=15)

    # RMSE
    axes[1].bar(summary_df["Model"], summary_df["RMSE_mean"],
                yerr=summary_df["RMSE_std"], capsize=5,
                color=colors[:len(summary_df)], alpha=0.85)
    for i, row in summary_df.iterrows():
        axes[1].text(i, row["RMSE_mean"] + row["RMSE_std"] + 0.002,
                     f'{row["RMSE_mean"]:.4f}', ha="center", va="bottom", fontsize=11)
    axes[1].set_title("LOYO Mean RMSE (±std)", fontsize=14)
    axes[1].set_ylabel("RMSE (m/yr)", fontsize=12)
    axes[1].tick_params(axis="x", labelsize=10, rotation=15)

    # MAE
    axes[2].bar(summary_df["Model"], summary_df["MAE_mean"],
                yerr=summary_df["MAE_std"], capsize=5,
                color=colors[:len(summary_df)], alpha=0.85)
    for i, row in summary_df.iterrows():
        axes[2].text(i, row["MAE_mean"] + row["MAE_std"] + 0.002,
                     f'{row["MAE_mean"]:.4f}', ha="center", va="bottom", fontsize=11)
    axes[2].set_title("LOYO Mean MAE (±std)", fontsize=14)
    axes[2].set_ylabel("MAE (m/yr)", fontsize=12)
    axes[2].tick_params(axis="x", labelsize=10, rotation=15)

    plt.suptitle("LOYO Cross-Validation — Model Comparison", fontsize=16, y=1.01)
    plt.tight_layout()
    plt.show()
    plt.close()

markers    = ["o", "s", "^", "D", "v"]
linestyles = ["-", "--", "-.", ":", (0, (3, 1, 1, 1))]

# ---- 图2: 逐年 R² 折线图 ----
if loyo_dfs:
    fig, ax = plt.subplots(figsize=(14, 6), dpi=100)
    for (model_name, df), marker, ls in zip(loyo_dfs.items(), markers, linestyles):
        ax.plot(df["year"], df["R2"], marker=marker, linestyle=ls,
                linewidth=2, markersize=6, label=model_name)
    ax.set_xlabel("Held-out Year", fontsize=13)
    ax.set_ylabel("R²", fontsize=13)
    ax.set_title("LOYO — Per-Year R² for All Models", fontsize=15)
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.close()

# ---- 图3: 逐年 RMSE 折线图 ----
if loyo_dfs:
    fig, ax = plt.subplots(figsize=(14, 6), dpi=100)
    for (model_name, df), marker, ls in zip(loyo_dfs.items(), markers, linestyles):
        ax.plot(df["year"], df["RMSE"], marker=marker, linestyle=ls,
                linewidth=2, markersize=6, label=model_name)
    ax.set_xlabel("Held-out Year", fontsize=13)
    ax.set_ylabel("RMSE (m/yr)", fontsize=13)
    ax.set_title("LOYO — Per-Year RMSE for All Models", fontsize=15)
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.close()

### 三方评估对比：LOYO vs 空间 GroupKFold CV vs Block CV

加载各模型的三种评估结果，生成综合汇总表 `cv_summary.csv` 及多维度可视化：
- **LOYO**（Leave-One-Year-Out）：每折留出一个年份，评估时间外推能力
- **Spatial CV**（5-fold GroupKFold by glacier）：按冰川分组，评估空间泛化能力
- **Block CV**（5×5 对角线）：冰川×年份双维度隔离，评估时空双重泛化能力

In [ ]:
MODEL_NAMES = ["RF", "LightGBM", "MLP/ANN", "Seasonal-LSTM", "Temporal-LSTM"]
MODEL_KEYS  = ["rf", "lgbm", "ann", "seasonal_lstm", "temporal_lstm"]

# LOYO CSV 文件名（各模型 LOYO 结果）
loyo_files = {
    "RF":            "rf_loyo_results.csv",
    "LightGBM":      "lgbm_loyo_results.csv",
    "MLP/ANN":       "ann_loyo_results.csv",
    "Seasonal-LSTM": "lstm_loyo_results.csv",
    "Temporal-LSTM": "temporal_lstm_loyo_results.csv",
}

# KFold CV / Block CV 文件名
kfold_files = {n: f"{k}_kfold_cv_results.csv" for n, k in zip(MODEL_NAMES, MODEL_KEYS)}
block_files = {n: f"{k}_block_cv_results.csv"  for n, k in zip(MODEL_NAMES, MODEL_KEYS)}


def load_cv(fname_dict, outputs_dir):
    out = {}
    for name, fname in fname_dict.items():
        fpath = os.path.join(outputs_dir, fname)
        if os.path.exists(fpath):
            out[name] = pd.read_csv(fpath)
            print(f"[OK] {name}: {fname}")
        else:
            print(f"[SKIP] {name}: {fname} not found")
    return out


print("── Loading LOYO ──")
loyo_dfs = load_cv(loyo_files, outputs_dir)
print("\n── Loading KFold CV ──")
kfold_dfs = load_cv(kfold_files, outputs_dir)
print("\n── Loading Block CV ──")
block_dfs = load_cv(block_files, outputs_dir)


# ── 汇总函数 ───────────────────────────────────────────────────────────
def summarize(df, metric_col="R2"):
    return float(df[metric_col].mean()), float(df[metric_col].std())


summary_rows = []
for name in MODEL_NAMES:
    row = {"Model": name}
    for method, dfs in [("LOYO", loyo_dfs),
                        ("KFold_CV", kfold_dfs),
                        ("Block_CV", block_dfs)]:
        if name in dfs:
            for metric in ["R2", "RMSE", "MAE"]:
                mn, sd = summarize(dfs[name], metric)
                row[f"{method}_{metric}_mean"] = round(mn, 4)
                row[f"{method}_{metric}_std"]  = round(sd, 4)
        else:
            for metric in ["R2", "RMSE", "MAE"]:
                row[f"{method}_{metric}_mean"] = float("nan")
                row[f"{method}_{metric}_std"]  = float("nan")
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
csv_path = os.path.join(outputs_dir, "cv_summary.csv")
summary_df.to_csv(csv_path, index=False)
print(f"\n[Summary] Saved → {csv_path}")
print(summary_df[[c for c in summary_df.columns
                  if c == "Model" or c.endswith("R2_mean")]].to_string(index=False))


# ── 图1：三方对比柱状图（R² / RMSE / MAE）────────────────────────────
colors_3 = {"LOYO": "#4CAF50", "KFold_CV": "#2196F3", "Block_CV": "#FF9800"}
methods  = ["LOYO", "KFold_CV", "Block_CV"]
labels   = ["LOYO", "KFold CV", "Block CV"]
models_present = [n for n in MODEL_NAMES if n in loyo_dfs or n in kfold_dfs or n in block_dfs]

for metric in ["R2", "RMSE", "MAE"]:
    fig, ax = plt.subplots(figsize=(14, 5), dpi=100)
    x = np.arange(len(models_present))
    width = 0.25
    for i, (method, label) in enumerate(zip(methods, labels)):
        means = [summary_df.loc[summary_df.Model == n, f"{method}_{metric}_mean"].values[0]
                 for n in models_present]
        stds  = [summary_df.loc[summary_df.Model == n, f"{method}_{metric}_std"].values[0]
                 for n in models_present]
        bars = ax.bar(x + (i - 1) * width, means, width, yerr=stds,
                      capsize=4, label=label, color=colors_3[method], alpha=0.85)
        for bar, v in zip(bars, means):
            if not np.isnan(v):
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                        f"{v:.3f}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(models_present, fontsize=10, rotation=10)
    ax.set_ylabel(metric, fontsize=12)
    ax.set_title(f"Model Comparison — {metric} (LOYO vs KFold CV vs Block CV)", fontsize=13)
    ax.legend(fontsize=11)
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.close()


# ── 图2：分方法 Grouped Bar（KFold CV vs Block CV 对比） ──────────────
for metric in ["R2", "RMSE", "MAE"]:
    fig, ax = plt.subplots(figsize=(12, 5), dpi=100)
    x = np.arange(len(models_present))
    width = 0.35
    for i, (method, label, color) in enumerate(zip(
            ["KFold_CV", "Block_CV"], ["KFold CV", "Block CV"],
            ["#2196F3", "#FF9800"])):
        means = [summary_df.loc[summary_df.Model == n, f"{method}_{metric}_mean"].values[0]
                 for n in models_present]
        stds  = [summary_df.loc[summary_df.Model == n, f"{method}_{metric}_std"].values[0]
                 for n in models_present]
        bars = ax.bar(x + (i - 0.5) * width, means, width, yerr=stds,
                      capsize=4, label=label, color=color, alpha=0.85)
        for bar, v in zip(bars, means):
            if not np.isnan(v):
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                        f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(models_present, fontsize=10, rotation=10)
    ax.set_ylabel(metric, fontsize=12)
    ax.set_title(f"KFold CV vs Block CV — {metric}", fontsize=13)
    ax.legend(fontsize=11)
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.close()


# ── 图3：Block CV 折间 R² 热力图（5×5，对角线有值）─────────────────────
fig, axes = plt.subplots(1, min(len(models_present), 5),
                         figsize=(4 * min(len(models_present), 5), 4), dpi=100)
if len(models_present) == 1:
    axes = [axes]

for ax, name in zip(axes if len(models_present) > 1 else [axes],
                    [n for n in models_present if n in block_dfs][:5]):
    df_b = block_dfs[name]
    mat  = np.full((5, 5), np.nan)
    for _, row_b in df_b.iterrows():
        f = int(row_b["fold"])
        if 0 <= f < 5:
            mat[f, f] = row_b["R2"]
    im = ax.imshow(mat, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    for i in range(5):
        if not np.isnan(mat[i, i]):
            ax.text(i, i, f"{mat[i,i]:.3f}", ha="center", va="center",
                    fontsize=10, fontweight="bold")
    ax.set_xticks(range(5)); ax.set_yticks(range(5))
    ax.set_xticklabels([f"Yr{i}" for i in range(5)], fontsize=8)
    ax.set_yticklabels([f"Gl{i}" for i in range(5)], fontsize=8)
    ax.set_title(f"{name}\nBlock CV R²", fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle("Block CV — Per-Fold R² Heatmap (diagonal only)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
plt.close()

print("\n[07_compare] 所有对比图已生成，cv_summary.csv 已保存。")
